## ResponseAPI tools loop with GPT-5.4-mini

In [1]:
import json

from rich.console import Console
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai = OpenAI()

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


def get_report():
    result = ""
    for index, todo in enumerate(todos):
        if completions[index]:
            result += f"Todo #{index+1}: [green][strike]{todo}[/strike][green]\n"
        else:
            result += f"Todo #{index+1}: {todo}\n"
    show(result)
    return result


def create_todos(args):
    descriptions = args.get('descriptions', [])
    todos.extend(descriptions)
    completions.extend([False] * len(descriptions))
    return get_report()


def mark_completed(args):
    index = args.get('index', 0)
    completion_notes = args.get('completion_notes', '')
    print(f"Get args: index={index} {type(index)} {len(completions)}\n---\n")
    if 1 <= index <= len(completions):
        completions[index-1] = True
        show(completion_notes)
    else:
        print(f"No completion by index: {index-1}")
    return get_report()

In [3]:
tools = [
    {
        'type': 'function',
        'name': 'create_todos',
        'description': 'Add new todos from a list descriptions and return the full list',
        'parameters': {
            'type': 'object',
            'properties': {
                'descriptions': {
                    'type': 'array',
                    'items': {'type': 'string'},
                }
            },
            'required': ['descriptions'],
        }
    },
    {
        'type': 'function',
        'name': 'mark_completed',
        'description': 'Mark competed the todo at given position (starting from 1) and return the full list',
        'parameters': {
            'type': 'object',
            'properties':{
                'index': {
                    'description': 'The 1-based index of the todo to mark as complete',
                    'type': 'integer',
                },
                'completion_notes': {
                    'description': 'Notes about how you completed the todo in rich console markup',
                    'type': 'string',
                }
                
            },
            'required': ['index', 'completion_notes'],
        }
    }  
]

system_prompt = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in return.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

In [4]:
def ask_gpt(input_list, instructions):
    done = False
    counter = 0
    while not done and counter < 10:
        counter += 1
        
        response = openai.responses.create(
            model='gpt-5.4-mini',
            tools=tools,
            input=input_list,
            instructions=instructions
        )
        print(f"Response:\n{response.output}\n---\n")
        input_list += response.output

        done = True
        for item in response.output:
            if item.type == 'function_call':
                done = False
                fn_name = item.name
                arguments = json.loads(item.arguments)
                print(f"Arguments: {arguments}\n---\n")
                fn = globals().get(fn_name)

                if fn:
                    print(f"Found function {fn_name}: {fn}\n---\n")
                    fn_output = fn(arguments)
                    input_list.append({
                        'type': 'function_call_output',
                        'call_id': item.call_id,
                        'output': fn_output,
                    })
                else:
                    done = True
                    print(f"Error: failed to find function {fn_name}")
                

    return response
            
            

In [ ]:
todos = []
completions = []
input_list = [{'role': 'user', 'content': user_message}]

resp = ask_gpt(input_list, system_prompt)

In [7]:
show(resp.output_text)

They meet at about 3:26 PM.

Quick check:
- Boston train gets a 1-hour head start: 60 miles
- Combined closing speed: 60 + 80 = 140 mph
- Time to meet after 3:00 PM: 60/140 = 3/7 hour ≈ 25.7 minutes

Answer: 3:26 PM